# 07 - Results Interpretation

**Purpose:** consolidate the final results around the project question:

> Can time-series history, social-network exposure, review-language signals, and stronger scikit-learn model families help forecast short-term shifts in community attention toward local Yelp businesses?

This interpretation uses the corrected forecasting cohort: **876 businesses** and **68,249 business-month rows** from the 2015-2021 modeling window, excluding rows before each business's first observed modeling-window review.

In [1]:
# Load final metrics, predictions, summaries, and lookup tables for interpretation.
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
PULSE_METRICS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_metrics.csv"
PULSE_PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_predictions.csv"
PULSE_TOPK_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_topk_metrics.csv"
PULSE_CALIBRATION_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_calibration.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"
FORECASTING_DATASET_PATH = PROCESSED_DIR / "forecasting_dataset.csv"
COHORT_BUSINESSES_PATH = PROCESSED_DIR / "forecasting_cohort_businesses.csv"
PULSE_PREDECESSOR_OUTPUT_PATH = OUTPUTS_DIR / "pulse_predecessor_analysis.csv"
CASE_STUDIES_OUTPUT_PATH = OUTPUTS_DIR / "attention_pulse_case_studies.csv"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
pulse_metrics = pd.read_csv(PULSE_METRICS_OUTPUT_PATH)
pulse_predictions = pd.read_csv(PULSE_PREDICTIONS_OUTPUT_PATH)
pulse_topk_metrics = pd.read_csv(PULSE_TOPK_OUTPUT_PATH)
pulse_calibration = pd.read_csv(PULSE_CALIBRATION_OUTPUT_PATH)
modeling = pd.read_csv(FORECASTING_DATASET_PATH)
business_lookup = pd.read_csv(COHORT_BUSINESSES_PATH)[["business_id", "name", "categories"]]
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,task,model,train_period,validation_period,test_period,rows,MAE,RMSE,WAPE,model_family,feature_count,model_name,business_count,scope
12,normal_pre_covid_test,review_count_regression_top10,Prophet,2015-01 to 2017-12,not used,2019-01 to 2019-12,120,14.370898,18.347806,0.230026,Prophet,0,Prophet,10.0,top_10_businesses_by_total_reviews
0,normal_pre_covid_test,review_count_regression,ML: HGB all modalities,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.299828,3.991996,0.380685,HistGradientBoostingRegressor,100,ML: HGB all modalities,NaN,NaN
1,normal_pre_covid_test,review_count_regression,ML: historical + business,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.307195,4.082504,0.381904,RandomForestRegressor,16,ML: historical + business,NaN,NaN
2,normal_pre_covid_test,review_count_regression,ML: all modalities,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.326933,4.091408,0.385171,RandomForestRegressor,100,ML: all modalities,NaN,NaN
3,normal_pre_covid_test,review_count_regression,ML: historical + SNA,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.344784,4.038973,0.388126,RandomForestRegressor,33,ML: historical + SNA,NaN,NaN
4,normal_pre_covid_test,review_count_regression,ML: historical + NLP,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.346847,4.069007,0.388468,RandomForestRegressor,71,ML: historical + NLP,NaN,NaN
5,normal_pre_covid_test,review_count_regression,ML: historical,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.349346,4.071374,0.388882,RandomForestRegressor,10,ML: historical,NaN,NaN
6,normal_pre_covid_test,review_count_regression,Baseline: rolling 3-month avg,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.356737,3.743842,0.390105,temporal_baseline,0,Baseline: rolling 3-month avg,NaN,NaN
7,normal_pre_covid_test,review_count_regression,ML: HGB selected top 20,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.368357,4.222579,0.392028,SelectKBest + HistGradientBoostingRegressor,100,ML: HGB selected top 20,NaN,NaN
8,normal_pre_covid_test,review_count_regression,Baseline: last month,2015-02 to 2017-12,2018-01 to 2018-12,2019-01 to 2019-12,10511,2.702217,4.265553,0.447291,temporal_baseline,0,Baseline: last month,NaN,NaN


## Regression Interpretation

Summarize which model family performs best in the single normal pre-COVID split and whether stronger alternatives improve on the simple temporal baselines. This pass compares Random Forests, HistGradientBoosting, Poisson count models, and selected-feature variants on the full business-month cohort. The scoped Prophet top-10 benchmark is kept out of the cohort-wide ranking.

In [2]:
# Build a compact regression summary by chronological split.
regression_summary_rows = []
for split_name, split_metrics in metrics[metrics["task"].eq("review_count_regression")].groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    hgb_all = split_metrics[split_metrics["model"] == "ML: HGB all modalities"].iloc[0]
    poisson_all = split_metrics[split_metrics["model"] == "ML: Poisson all modalities"].iloc[0]
    selected_hgb = split_metrics[split_metrics["model"] == "ML: HGB selected top 20"].iloc[0]
    regression_summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_family": best["model_family"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "business_WAPE": business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "nlp_WAPE": nlp["WAPE"],
        "all_modalities_WAPE": all_modalities["WAPE"],
        "hgb_all_WAPE": hgb_all["WAPE"],
        "poisson_all_WAPE": poisson_all["WAPE"],
        "selected_hgb_WAPE": selected_hgb["WAPE"],
        "best_vs_last_month_relative_change": (best["WAPE"] - split_metrics[split_metrics["model"] == "Baseline: last month"].iloc[0]["WAPE"]) / split_metrics[split_metrics["model"] == "Baseline: last month"].iloc[0]["WAPE"],
        "all_vs_historical_relative_change": (all_modalities["WAPE"] - hist["WAPE"]) / hist["WAPE"],
        "all_vs_business_relative_change": (all_modalities["WAPE"] - business["WAPE"]) / business["WAPE"],
    })
regression_summary = pd.DataFrame(regression_summary_rows)
regression_summary

,split,best_model,best_family,best_WAPE,historical_WAPE,business_WAPE,sna_WAPE,nlp_WAPE,all_modalities_WAPE,hgb_all_WAPE,poisson_all_WAPE,selected_hgb_WAPE,best_vs_last_month_relative_change,all_vs_historical_relative_change,all_vs_business_relative_change
0,normal_pre_covid_test,ML: HGB all modalities,HistGradientBoostingRegressor,0.380685,0.388882,0.381904,0.388126,0.388468,0.385171,0.380685,0.449751,0.392028,-0.148911,-0.00954,0.008555


## Pulse Interpretation

Summarize pulse-classification performance with class balance and probability quality in mind:

- F1 uses a cutoff tuned on the validation period;
- PR-AUC and top-k metrics assess ranking quality for rare pulse events;
- Brier score and calibration bins assess whether pulse probabilities behave like useful risk estimates.

In [3]:
# Build a compact pulse-classification summary by chronological split.
# Pulse summaries emphasize rare-event detection and probability quality rather than overall accuracy.
pulse_summary_rows = []
for split_name, split_metrics in pulse_metrics.groupby("split"):
    ranked = split_metrics.sort_values(["F1", "PR_AUC"], ascending=[False, False]).reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + SNA"].iloc[0]
    nlp = split_metrics[split_metrics["model"] == "ML: historical + NLP"].iloc[0]
    all_modalities = split_metrics[split_metrics["model"] == "ML: all modalities"].iloc[0]
    hgb_all = split_metrics[split_metrics["model"] == "ML: HGB all modalities"].iloc[0]
    logistic_all = split_metrics[split_metrics["model"] == "ML: Logistic all modalities"].iloc[0]
    selected_hgb = split_metrics[split_metrics["model"] == "ML: HGB selected top 20"].iloc[0]
    best_brier = split_metrics.sort_values("Brier").iloc[0]

    split_topk_10 = pulse_topk_metrics[
        (pulse_topk_metrics["split"] == split_name)
        & (pulse_topk_metrics["k_fraction"] == 0.10)
    ].copy()
    best_topk_10 = split_topk_10.sort_values(["precision_at_k", "recall_at_k"], ascending=[False, False]).iloc[0]
    all_topk_10 = split_topk_10[split_topk_10["model"] == "ML: all modalities"].iloc[0]

    pulse_summary_rows.append({
        "split": split_name,
        "positive_rate": all_modalities["positive_rate"],
        "best_model": best["model"],
        "best_family": best["model_family"],
        "best_F1": best["F1"],
        "best_PR_AUC": best["PR_AUC"],
        "best_Brier": best["Brier"],
        "best_threshold": best["decision_threshold"],
        "best_validation_F1": best["validation_F1"],
        "best_brier_model": best_brier["model"],
        "best_brier": best_brier["Brier"],
        "historical_F1": hist["F1"],
        "business_F1": business["F1"],
        "sna_F1": sna["F1"],
        "nlp_F1": nlp["F1"],
        "all_modalities_F1": all_modalities["F1"],
        "all_modalities_PR_AUC": all_modalities["PR_AUC"],
        "hgb_all_F1": hgb_all["F1"],
        "hgb_all_PR_AUC": hgb_all["PR_AUC"],
        "logistic_all_F1": logistic_all["F1"],
        "logistic_all_PR_AUC": logistic_all["PR_AUC"],
        "selected_hgb_F1": selected_hgb["F1"],
        "selected_hgb_PR_AUC": selected_hgb["PR_AUC"],
        "selected_hgb_Brier": selected_hgb["Brier"],
        "all_modalities_threshold": all_modalities["decision_threshold"],
        "best_precision_at_10pct_model": best_topk_10["model"],
        "best_precision_at_10pct": best_topk_10["precision_at_k"],
        "best_recall_at_10pct": best_topk_10["recall_at_k"],
        "all_modalities_precision_at_10pct": all_topk_10["precision_at_k"],
        "all_modalities_recall_at_10pct": all_topk_10["recall_at_k"],
    })
pulse_summary = pd.DataFrame(pulse_summary_rows)
pulse_summary

,split,positive_rate,best_model,best_family,best_F1,best_PR_AUC,best_Brier,best_threshold,best_validation_F1,best_brier_model,...,logistic_all_PR_AUC,selected_hgb_F1,selected_hgb_PR_AUC,selected_hgb_Brier,all_modalities_threshold,best_precision_at_10pct_model,best_precision_at_10pct,best_recall_at_10pct,all_modalities_precision_at_10pct,all_modalities_recall_at_10pct
0,normal_pre_covid_test,0.131957,ML: HGB selected top 20,SelectKBest + HistGradientBoostingClassifier,0.30869,0.237247,0.110175,0.155,0.382133,ML: HGB all modalities,...,0.210482,0.30869,0.237247,0.110175,0.4,ML: historical,0.301331,0.228551,0.292776,0.222062


## Final Model Comparison Summary

This table is the report-ready bridge between metrics and interpretation for the normal pre-COVID split. It combines the best full-cohort review-count model, the best attention-pulse model, simple baseline comparisons, and a compact modality takeaway.

In [4]:
# Combine regression, pulse, baseline, and modality comparisons into one final table.
comparison_rows = []
for split_name in regression_summary["split"]:
    split_regression = metrics[(metrics["split"] == split_name) & metrics["task"].eq("review_count_regression")].copy()
    split_pulse = pulse_metrics[pulse_metrics["split"] == split_name].copy()
    reg = regression_summary[regression_summary["split"] == split_name].iloc[0]
    pulse = pulse_summary[pulse_summary["split"] == split_name].iloc[0]

    last_month = split_regression[split_regression["model"] == "Baseline: last month"].iloc[0]
    best_pulse_baseline = (
        split_pulse[split_pulse["model_family"] == "rule_baseline"]
        .sort_values(["F1", "PR_AUC"], ascending=[False, False])
        .iloc[0]
    )

    sna_wape_change = (reg["sna_WAPE"] - reg["historical_WAPE"]) / reg["historical_WAPE"]
    nlp_wape_change = (reg["nlp_WAPE"] - reg["historical_WAPE"]) / reg["historical_WAPE"]
    pulse_sna_f1_change = pulse["sna_F1"] - pulse["historical_F1"]
    pulse_nlp_f1_change = pulse["nlp_F1"] - pulse["historical_F1"]

    modality_takeaway = "Normal pre-COVID patterns are learnable; all-modality HGB helps count forecasting, while selected HGB gives the cleanest pulse F1."

    comparison_rows.append({
        "split": split_name,
        "best_review_count_model": reg["best_model"],
        "best_review_count_WAPE": reg["best_WAPE"],
        "last_month_baseline_WAPE": last_month["WAPE"],
        "best_count_vs_last_month_pct": reg["best_vs_last_month_relative_change"] * 100,
        "best_pulse_model": pulse["best_model"],
        "best_pulse_F1": pulse["best_F1"],
        "best_pulse_PR_AUC": pulse["best_PR_AUC"],
        "best_pulse_baseline": best_pulse_baseline["model"],
        "best_pulse_baseline_F1": best_pulse_baseline["F1"],
        "best_pulse_vs_baseline_F1_delta": pulse["best_F1"] - best_pulse_baseline["F1"],
        "best_top10_precision_model": pulse["best_precision_at_10pct_model"],
        "best_top10_precision": pulse["best_precision_at_10pct"],
        "sna_vs_historical_WAPE_pct": sna_wape_change * 100,
        "nlp_vs_historical_WAPE_pct": nlp_wape_change * 100,
        "pulse_sna_vs_historical_F1_delta": pulse_sna_f1_change,
        "pulse_nlp_vs_historical_F1_delta": pulse_nlp_f1_change,
        "modality_takeaway": modality_takeaway,
    })

model_comparison_summary = pd.DataFrame(comparison_rows)
MODEL_COMPARISON_SUMMARY_PATH = OUTPUTS_DIR / "model_comparison_summary.csv"
model_comparison_summary.to_csv(MODEL_COMPARISON_SUMMARY_PATH, index=False)
model_comparison_summary

,split,best_review_count_model,best_review_count_WAPE,last_month_baseline_WAPE,best_count_vs_last_month_pct,best_pulse_model,best_pulse_F1,best_pulse_PR_AUC,best_pulse_baseline,best_pulse_baseline_F1,best_pulse_vs_baseline_F1_delta,best_top10_precision_model,best_top10_precision,sna_vs_historical_WAPE_pct,nlp_vs_historical_WAPE_pct,pulse_sna_vs_historical_F1_delta,pulse_nlp_vs_historical_F1_delta,modality_takeaway
0,normal_pre_covid_test,ML: HGB all modalities,0.380685,0.447291,-14.891057,ML: HGB selected top 20,0.30869,0.237247,Baseline: rising recent activity,0.215558,0.093132,ML: historical,0.301331,-0.194192,-0.106366,0.002314,-0.00442,Normal pre-COVID patterns are learnable; all-m...


## Community-Level vs Business-Level Forecasting

Community-level forecasting is generally more predictable in the sense that aggregation smooths out individual-business noise: the top reviewer communities show clearer long-run trends and repeated seasonal movement than most individual businesses. The Prophet community evaluation uses the same normal pre-COVID holdout as the business-level Prophet benchmark, so it reads as a cleaner rhythm test rather than a pandemic-shock robustness test.

Business-level forecasting is more actionable. A community forecast can say that a reviewer group is becoming more or less active, but it cannot tell a restaurant, bar, tour operator, or analyst which specific business-month is likely to gain attention. The business-level models and pulse classifier answer the operational version of the question: where might community attention concentrate next?

For the research question, the two granularities are complementary. Community-level models support the idea that Yelp attention has social-group rhythms, while business-level models test whether those rhythms, recent activity, sentiment, and SNA exposure can be localized into useful forecasts. The practical conclusion is that community-level forecasting is best for context, seasonality, and shock diagnostics, while business-level forecasting is the better unit for targeted interpretation and decision-making.

## Random Forest Feature Importance

The cells below inspect the all-modality Random Forest models used for the review-count and attention-pulse tasks. They reuse `rf_model`, `rf_clf`, `feature_cols`, and the normal pre-COVID train/test split when those objects already exist; when the notebook is run from a clean kernel, they rebuild the same split and fit equivalent Random Forest models so the interpretation is reproducible.

In [ ]:
# Extract and plot top Random Forest feature importances for regression and pulse detection.
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance

import matplotlib
IN_NOTEBOOK = "get_ipython" in globals()
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURES_DIR = OUTPUTS_DIR / "figures" / "models"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

historical_features = [
    "prev_month_reviews",
    "rolling_3_avg",
    "rolling_6_avg",
    "same_month_previous_year",
    "cumulative_review_count",
    "avg_rating_history",
    "calendar_month",
    "month_sin",
    "month_cos",
    "feature_month_number",
]
business_features = [
    "business_stars",
    "business_review_count",
    "is_restaurant",
    "is_food",
    "is_nightlife",
    "is_tourism",
]
static_sna_features = [
    "recent_reviewer_count",
    "avg_reviewer_degree",
    "avg_reviewer_weighted_degree",
    "max_reviewer_pagerank",
    "max_reviewer_weighted_pagerank",
    "avg_reviewer_weighted_pagerank",
    "sum_reviewer_weighted_pagerank",
    "sum_reviewer_weighted_degree",
    "avg_reviewer_friend_shared_businesses",
    "avg_reviewer_friend_category_overlap",
    "fraction_connected_reviewers",
    "fraction_weighted_connected_reviewers",
    "fraction_repeat_reviewers",
    "reviewer_community_diversity",
]
dynamic_social_exposure_features = [
    "fraction_active_to_date_reviewers",
    "avg_reviewer_reviews_to_date",
    "max_reviewer_reviews_to_date",
    "avg_reviewer_active_months_to_date",
    "fraction_largest_component_reviewers",
    "fraction_high_weighted_pagerank_reviewers",
    "fraction_high_weighted_degree_reviewers",
    "avg_reviewer_recency_months",
    "fraction_current_month_reviewers",
]
sna_features = static_sna_features + dynamic_social_exposure_features
nlp_features = [
    "recent_text_review_count",
    "avg_recent_review_char_count",
    "avg_recent_review_word_count",
    "share_recent_positive_language",
    "share_recent_negative_language",
    "avg_recent_positive_word_hits",
    "avg_recent_negative_word_hits",
    "recent_text_avg_stars",
    "sentiment_compound_mean",
    "sentiment_positive_pct",
    "sentiment_negative_pct",
] + sorted([column for column in modeling.columns if column.startswith("tfidf_recent_")])

if "feature_cols" not in globals():
    feature_cols = historical_features + business_features + sna_features + nlp_features
feature_cols = list(feature_cols)

missing_feature_cols = [column for column in feature_cols if column not in modeling.columns]
if missing_feature_cols:
    raise ValueError(f"Missing feature columns: {missing_feature_cols}")

SPLIT_NAME = "normal_pre_covid_test"
if "splits" in globals() and SPLIT_NAME in splits:
    feature_importance_split = splits[SPLIT_NAME]
else:
    feature_importance_split = {
        "train_start": "2015-02",
        "train_end": "2017-12",
        "validation_start": "2018-01",
        "validation_end": "2018-12",
        "test_start": "2019-01",
        "test_end": "2019-12",
    }

if "split_modeling_data" in globals():
    train_df, validation_df, test_df = split_modeling_data(feature_importance_split)
else:
    train_mask = (
        (modeling["target_month_str"] >= feature_importance_split["train_start"])
        & (modeling["target_month_str"] <= feature_importance_split["train_end"])
    )
    validation_mask = (
        (modeling["target_month_str"] >= feature_importance_split["validation_start"])
        & (modeling["target_month_str"] <= feature_importance_split["validation_end"])
    )
    test_mask = (
        (modeling["target_month_str"] >= feature_importance_split["test_start"])
        & (modeling["target_month_str"] <= feature_importance_split["test_end"])
    )
    train_df = modeling[train_mask].copy()
    validation_df = modeling[validation_mask].copy()
    test_df = modeling[test_mask].copy()

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train_regression = train_df["target_next_month_reviews"].astype(float)
y_test_regression = test_df["target_next_month_reviews"].astype(float)
y_train_pulse = train_df["attention_pulse"].astype(int)
y_test_pulse = test_df["attention_pulse"].astype(int)

if "rf_model" not in globals():
    rf_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )
    rf_model.fit(X_train, y_train_regression)

if "rf_clf" not in globals():
    rf_clf = RandomForestClassifier(
        n_estimators=140,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    rf_clf.fit(X_train, y_train_pulse)

feature_type_lookup = {}
for feature in historical_features:
    feature_type_lookup[feature] = "Lag features"
for feature in sna_features:
    feature_type_lookup[feature] = "SNA features"
for feature in nlp_features:
    feature_type_lookup[feature] = "NLP/sentiment features"
for feature in business_features:
    feature_type_lookup[feature] = "Business metadata features"

feature_type_colors = {
    "Lag features": "#4C78A8",
    "SNA features": "#F58518",
    "NLP/sentiment features": "#54A24B",
    "Business metadata features": "#8C8C8C",
}


def build_importance_frame(model, model_label):
    if len(model.feature_importances_) != len(feature_cols):
        raise ValueError(
            f"{model_label} has {len(model.feature_importances_)} importances, "
            f"but feature_cols has {len(feature_cols)} columns."
        )
    frame = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_,
    })
    frame["feature_type"] = frame["feature"].map(feature_type_lookup).fillna("Business metadata features")
    frame["model"] = model_label
    frame["impurity_rank"] = frame["importance"].rank(method="first", ascending=False).astype(int)
    return frame.sort_values("importance", ascending=False).reset_index(drop=True)


def plot_top_importances(importance_frame, title, output_path):
    top_features = importance_frame.head(20).sort_values("importance", ascending=True)
    colors = top_features["feature_type"].map(feature_type_colors)
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top_features["feature"], top_features["importance"], color=colors)
    ax.set_title(title)
    ax.set_xlabel("Impurity-based feature importance")
    ax.set_ylabel("")
    legend_handles = [
        plt.Rectangle((0, 0), 1, 1, color=color, label=label)
        for label, color in feature_type_colors.items()
    ]
    ax.legend(handles=legend_handles, loc="lower right", frameon=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    if IN_NOTEBOOK:
        plt.show()
    plt.close(fig)


regression_importance = build_importance_frame(rf_model, "Review-count Random Forest")
pulse_importance = build_importance_frame(rf_clf, "Pulse Random Forest")

REGRESSION_IMPORTANCE_PATH = FIGURES_DIR / "feature_importance_regression.png"
PULSE_IMPORTANCE_PATH = FIGURES_DIR / "feature_importance_pulse.png"
plot_top_importances(
    regression_importance,
    "Top 20 Random Forest Feature Importances - Review Count Regression",
    REGRESSION_IMPORTANCE_PATH,
)
plot_top_importances(
    pulse_importance,
    "Top 20 Random Forest Feature Importances - Attention Pulse Classification",
    PULSE_IMPORTANCE_PATH,
)

print(REGRESSION_IMPORTANCE_PATH)
print(PULSE_IMPORTANCE_PATH)
pd.concat([regression_importance.head(20), pulse_importance.head(20)], ignore_index=True)


### Interpreting the Top Features

For the review-count regressor, the top five features are `rolling_3_avg`, `recent_reviewer_count`, `rolling_6_avg`, `recent_text_review_count`, and `business_review_count`. This makes intuitive sense: next-month review volume is anchored by recent momentum, the breadth of the recent reviewer base, recent text-bearing review activity, and the business's longer-run popularity. The SNA signal `recent_reviewer_count` being near the top is especially useful because it suggests the model is not only learning how many reviews arrived recently, but also how many distinct people were actively participating around the business.

For the attention-pulse classifier, the top five features are `cumulative_review_count`, `month_sin`, `rolling_6_avg`, `business_review_count`, and `calendar_month`. These point to scale, seasonality, and the business's normal review baseline. That is also plausible for pulse detection: unusual attention is easier to identify once the model knows whether a business is generally high-volume, what its recent baseline looks like, and where the target month falls in New Orleans' seasonal/tourism cycle.

In [ ]:
# Compare impurity-based Random Forest importances with permutation importance on the regression test set.
permutation_result = permutation_importance(
    rf_model,
    X_test,
    y_test_regression,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)

permutation_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "permutation_importance_mean": permutation_result.importances_mean,
    "permutation_importance_std": permutation_result.importances_std,
})
permutation_importance_df["permutation_rank"] = (
    permutation_importance_df["permutation_importance_mean"]
    .rank(method="first", ascending=False)
    .astype(int)
)

importance_comparison = regression_importance.merge(permutation_importance_df, on="feature", how="left")
importance_comparison["rank_delta_permutation_minus_impurity"] = (
    importance_comparison["permutation_rank"] - importance_comparison["impurity_rank"]
)
importance_comparison = importance_comparison.sort_values("permutation_rank").reset_index(drop=True)
importance_comparison[[
    "feature",
    "feature_type",
    "importance",
    "impurity_rank",
    "permutation_importance_mean",
    "permutation_importance_std",
    "permutation_rank",
    "rank_delta_permutation_minus_impurity",
]].head(20)


### Permutation Importance Check

Permutation importance on the regression test set broadly agrees with the impurity ranking: the same five variables dominate, with only a small order change. `recent_reviewer_count` moves from impurity rank 2 to permutation rank 1, while `rolling_3_avg` moves from impurity rank 1 to permutation rank 2; `rolling_6_avg` remains rank 3. This reinforces the interpretation that recent review momentum and the size of the recent reviewer pool are the strongest signals for next-month activity.

The main discrepancies are lower in the ranking. `business_review_count` rises ahead of `recent_text_review_count` under permutation importance, suggesting long-run popularity contributes more to held-out predictive performance than the impurity split score alone implies. `fraction_current_month_reviewers` appears in the impurity top 10 but falls outside the permutation top 20, which is a useful warning that impurity-based Random Forest importances can overstate correlated or high-split-opportunity features. Overall, the model story is stable: recent activity, reviewer breadth, and business scale matter most.

## What Precedes An Attention Pulse?

This comparison checks whether pulse rows have different recent conditions than non-pulse rows before the target month arrives. It focuses on interpretable signals: review momentum, recent text volume/language, reviewer centrality, and social exposure.


In [5]:
# Print concise text summaries for the report narrative.
pre_pulse_features = [
    "prev_month_reviews",
    "rolling_3_avg",
    "rolling_6_avg",
    "same_month_previous_year",
    "recent_text_review_count",
    "avg_recent_review_word_count",
    "share_recent_positive_language",
    "share_recent_negative_language",
    "recent_text_avg_stars",
    "sentiment_compound_mean",
    "sentiment_positive_pct",
    "sentiment_negative_pct",
    "recent_reviewer_count",
    "avg_reviewer_weighted_degree",
    "max_reviewer_weighted_pagerank",
    "fraction_repeat_reviewers",
    "reviewer_community_diversity",
    "fraction_active_to_date_reviewers",
    "avg_reviewer_recency_months",
] + sorted([column for column in modeling.columns if column.startswith("tfidf_recent_")])[:20]

available_pre_pulse_features = [feature for feature in pre_pulse_features if feature in modeling.columns]
# Compare feature averages to identify conditions that tend to precede pulses.
pulse_predecessor_summary = (
    modeling.groupby("attention_pulse")[available_pre_pulse_features]
    .mean()
    .T
    .rename(columns={0: "non_pulse_mean", 1: "pulse_mean"})
)
pulse_predecessor_summary["absolute_difference"] = pulse_predecessor_summary["pulse_mean"] - pulse_predecessor_summary["non_pulse_mean"]
pulse_predecessor_summary["relative_lift_vs_non_pulse"] = pulse_predecessor_summary["pulse_mean"] / pulse_predecessor_summary["non_pulse_mean"].replace(0, np.nan)
pulse_predecessor_summary = pulse_predecessor_summary.sort_values("absolute_difference", ascending=False)
pulse_predecessor_summary_output = pulse_predecessor_summary.reset_index().rename(columns={"index": "feature"})
pulse_predecessor_summary_output.to_csv(PULSE_PREDECESSOR_OUTPUT_PATH, index=False)
pulse_predecessor_summary_output.head(20)

attention_pulse,feature,non_pulse_mean,pulse_mean,absolute_difference,relative_lift_vs_non_pulse
0,sentiment_positive_pct,70.660392,80.419926,9.759534,1.138119
1,avg_recent_review_word_count,87.215370,94.201212,6.985842,1.080099
2,avg_reviewer_weighted_degree,31.289212,34.597184,3.307972,1.105722
3,sentiment_negative_pct,9.219162,10.181082,0.961920,1.104339
4,recent_text_avg_stars,3.655975,3.937210,0.281235,1.076925
5,prev_month_reviews,5.010349,5.145895,0.135546,1.027053
6,sentiment_compound_mean,0.556517,0.638881,0.082364,1.147999
7,reviewer_community_diversity,2.118906,2.181838,0.062932,1.029700
8,share_recent_positive_language,0.721067,0.779101,0.058034,1.080483
9,tfidf_recent_amazing,0.227545,0.254093,0.026548,1.116670


## Example Pulse Case Studies

The table below selects a few high-signal business-months from the normal pre-COVID 2019 test split. It includes correct and incorrect pulse predictions so the report can discuss what the model sees, where it succeeds, and where community attention remains hard to anticipate.

In [6]:
# Compare feature means before pulse vs non-pulse outcomes.
case_model = "ML: HGB selected top 20"
case_split = "normal_pre_covid_test"
case_pool = pulse_predictions[
    (pulse_predictions["split"] == case_split)
    & (pulse_predictions["model"] == case_model)
].copy()
if case_pool.empty:
    case_model = "ML: all modalities"
    case_pool = pulse_predictions[
        (pulse_predictions["split"] == case_split)
        & (pulse_predictions["model"] == case_model)
    ].copy()

# Label prediction outcomes so the case studies include successes and errors.
case_pool["case_type"] = np.select(
    [
        case_pool["attention_pulse"].eq(1) & case_pool["prediction"].eq(1),
        case_pool["attention_pulse"].eq(1) & case_pool["prediction"].eq(0),
        case_pool["attention_pulse"].eq(0) & case_pool["prediction"].eq(1),
        case_pool["attention_pulse"].eq(0) & case_pool["prediction"].eq(0),
    ],
    ["true_positive_pulse", "missed_pulse", "false_alarm", "true_negative"],
    default="other",
)

case_specs = [
    ("true_positive_pulse", False),
    ("missed_pulse", False),
    ("false_alarm", False),
    ("true_positive_pulse", True),
    ("missed_pulse", True),
]
case_frames = []
used_index = set()
for case_type, ascending_probability in case_specs:
    candidates = case_pool[case_pool["case_type"] == case_type].copy()
    if candidates.empty:
        continue
    candidates = candidates.sort_values(
        ["probability", "target_next_month_reviews"],
        ascending=[ascending_probability, False],
    )
    for idx, row in candidates.iterrows():
        if idx not in used_index:
            used_index.add(idx)
            case_frames.append(row.to_frame().T)
            break

case_studies = pd.concat(case_frames, ignore_index=True) if case_frames else case_pool.head(5)
case_studies = (
    case_studies
    .merge(business_lookup, on="business_id", how="left")
    .merge(
        modeling[[
            "business_id",
            "feature_month_str",
            "rolling_3_avg",
            "rolling_6_avg",
            "recent_text_review_count",
            "share_recent_positive_language",
            "share_recent_negative_language",
            "sentiment_compound_mean",
            "sentiment_positive_pct",
            "sentiment_negative_pct",
            "recent_reviewer_count",
            "avg_reviewer_weighted_degree",
            "max_reviewer_weighted_pagerank",
            "fraction_repeat_reviewers",
            "reviewer_community_diversity",
        ]],
        on=["business_id", "feature_month_str"],
        how="left",
    )
)

case_studies["interpretation_note"] = np.select(
    [
        case_studies["case_type"].eq("true_positive_pulse"),
        case_studies["case_type"].eq("missed_pulse"),
        case_studies["case_type"].eq("false_alarm"),
    ],
    [
        "Model correctly flagged a future pulse; inspect recent momentum, reviewers, and language as plausible warning signals.",
        "A pulse occurred despite a low model decision; useful for discussing hidden offline events or weak digital precursors.",
        "Model expected a pulse that did not materialize; useful for discussing noisy attention signals and threshold trade-offs.",
    ],
    default="Stable non-pulse example.",
)

case_columns = [
    "case_type",
    "model",
    "business_id",
    "name",
    "target_month_str",
    "attention_pulse",
    "prediction",
    "probability",
    "decision_threshold",
    "target_next_month_reviews",
    "pulse_baseline_reviews",
    "pulse_relative_lift",
    "rolling_3_avg",
    "rolling_6_avg",
    "recent_text_review_count",
    "share_recent_positive_language",
    "share_recent_negative_language",
    "sentiment_compound_mean",
    "sentiment_positive_pct",
    "sentiment_negative_pct",
    "recent_reviewer_count",
    "avg_reviewer_weighted_degree",
    "max_reviewer_weighted_pagerank",
    "fraction_repeat_reviewers",
    "reviewer_community_diversity",
    "interpretation_note",
]
case_studies = case_studies[case_columns]
case_studies.to_csv(CASE_STUDIES_OUTPUT_PATH, index=False)
case_studies

,case_type,model,business_id,name,target_month_str,attention_pulse,prediction,probability,decision_threshold,target_next_month_reviews,...,share_recent_negative_language,sentiment_compound_mean,sentiment_positive_pct,sentiment_negative_pct,recent_reviewer_count,avg_reviewer_weighted_degree,max_reviewer_weighted_pagerank,fraction_repeat_reviewers,reviewer_community_diversity,interpretation_note
0,true_positive_pulse,ML: HGB selected top 20,5xRmemqmR89BpGoOA25hqA,Dian Xin,2019-03,1,1,0.914727,0.155,15.0,...,0.208333,0.812417,95.652174,4.347826,24.0,75.845816,0.001291,0.666667,4.0,Model correctly flagged a future pulse; inspec...
1,missed_pulse,ML: HGB selected top 20,OKpM0cvlI7rb5j_K1XDa7A,Southern Style Tours,2019-08,1,0,0.154648,0.155,4.0,...,0.000000,0.991900,100.000000,0.000000,1.0,5.509547,0.000019,0.000000,1.0,A pulse occurred despite a low model decision;...
2,false_alarm,ML: HGB selected top 20,t1qF12NdW8KvCqxqbvy-Hg,Cafe Envie & Espresso Bar,2019-04,0,1,0.732438,0.155,0.0,...,0.000000,0.041400,50.000000,50.000000,3.0,2.644683,0.000022,0.666667,1.0,Model expected a pulse that did not materializ...
3,true_positive_pulse,ML: HGB selected top 20,o0e5bWndZc7TNiYF2D7eIg,Marie Laveau House of Voodoo,2019-04,1,1,0.155087,0.155,5.0,...,0.545455,-0.242520,20.000000,80.000000,11.0,2.326358,0.000036,0.727273,2.0,Model correctly flagged a future pulse; inspec...
4,missed_pulse,ML: HGB selected top 20,YvrV_Tc9syF2Wch0HOIT3A,Flamingo A-Go-Go,2019-08,1,0,0.016963,0.155,39.0,...,0.341463,0.605332,78.947368,21.052632,81.0,3.591906,0.000186,0.802469,5.0,A pulse occurred despite a low model decision;...


In [7]:
# Print social graph and forecasting dataset metadata for auditability.
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
threshold_candidates: [2, 3, 5, 10, 20]
edge_weight_formula: 1 + log1p(shared_business_count) + category_jaccard
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
mean_edge_weight: 1.833072733525831
mean_edge_shared_business_count: 2.048550936014688
mean_edge_category_jaccard: 0.2671618532172656
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: weighted_louvain_largest_component
communities_assigned: 63
threshold_sensitivity_output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\active_reviewer_threshold_sensitivity.csv
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 876
row_count: 68249
feature_month_min: 2015-01
feature

In [8]:
# Select representative true positive, missed, and false-alarm pulse case studies.
for _, row in regression_summary.iterrows():
    split = row["split"]
    best_vs_last_month = row["best_vs_last_month_relative_change"] * 100
    all_vs_hist = row["all_vs_historical_relative_change"] * 100
    all_vs_business = row["all_vs_business_relative_change"] * 100
    print(f"{split} regression:")
    print(f"  Best model: {row['best_model']} ({row['best_family']}) with WAPE={row['best_WAPE']:.4f}")
    print(f"  Best vs last-month baseline: {best_vs_last_month:+.2f}%")
    print(f"  HGB all modalities WAPE: {row['hgb_all_WAPE']:.4f}")
    print(f"  Poisson all modalities WAPE: {row['poisson_all_WAPE']:.4f}")
    print(f"  Selected HGB WAPE: {row['selected_hgb_WAPE']:.4f}")
    print(f"  All modalities vs historical: {all_vs_hist:+.2f}%")
    print(f"  All modalities vs historical+business: {all_vs_business:+.2f}%")

print()
for _, row in pulse_summary.iterrows():
    split = row["split"]
    print(f"{split} attention pulses:")
    print(f"  Positive rate: {row['positive_rate']:.3f}")
    print(f"  Best thresholded model: {row['best_model']} ({row['best_family']}) with F1={row['best_F1']:.4f}, PR-AUC={row['best_PR_AUC']:.4f}, Brier={row['best_Brier']:.4f}")
    print(f"  Best model threshold: {row['best_threshold']:.3f}; validation F1={row['best_validation_F1']:.4f}")
    print(f"  Best Brier model: {row['best_brier_model']} with Brier={row['best_brier']:.4f}")
    print(f"  HGB selected top 20: F1={row['selected_hgb_F1']:.4f}, PR-AUC={row['selected_hgb_PR_AUC']:.4f}, Brier={row['selected_hgb_Brier']:.4f}")
    print(f"  Logistic all modalities: F1={row['logistic_all_F1']:.4f}, PR-AUC={row['logistic_all_PR_AUC']:.4f}")
    print(f"  Best precision@10%: {row['best_precision_at_10pct_model']} with precision={row['best_precision_at_10pct']:.4f}, recall={row['best_recall_at_10pct']:.4f}")

normal_pre_covid_test regression:
  Best model: ML: HGB all modalities (HistGradientBoostingRegressor) with WAPE=0.3807
  Best vs last-month baseline: -14.89%
  HGB all modalities WAPE: 0.3807
  Poisson all modalities WAPE: 0.4498
  Selected HGB WAPE: 0.3920
  All modalities vs historical: -0.95%
  All modalities vs historical+business: +0.86%

normal_pre_covid_test attention pulses:
  Positive rate: 0.132
  Best thresholded model: ML: HGB selected top 20 (SelectKBest + HistGradientBoostingClassifier) with F1=0.3087, PR-AUC=0.2372, Brier=0.1102
  Best model threshold: 0.155; validation F1=0.3821
  Best Brier model: ML: HGB all modalities with Brier=0.1100
  HGB selected top 20: F1=0.3087, PR-AUC=0.2372, Brier=0.1102
  Logistic all modalities: F1=0.2867, PR-AUC=0.2105
  Best precision@10%: ML: historical with precision=0.3013, recall=0.2286


## Interpretation Structure

1. **Normal-period forecasting:** the project now estimates ordinary monthly review activity using a 2019 pre-COVID test year, avoiding the 2020-2021 structural break as the main evaluation target.
2. **Count learnability:** in the 2019 test split, the all-modality HistGradientBoosting regressor is the best full-cohort count model, improving on the last-month baseline.
3. **Attention-pulse framing:** pulse prediction better matches the project objective because it asks whether a business is about to receive unusual community attention, not merely whether already-popular businesses keep receiving reviews.
4. **Sentiment as precursor, not magic lift:** pulse rows show higher same-month VADER positivity and compound sentiment than non-pulse rows, but the NLP-only model does not dominate thresholded pulse F1 after temporal/business signals are included.
5. **Ranking versus thresholding:** selected HGB gives the best thresholded pulse F1, while top-k metrics show which models retrieve denser candidate sets for analyst review.
6. **Multimodal analytics:** social exposure, recent language, VADER sentiment, and TF-IDF terms add interpretability and auditability. Predictively, more modalities are useful selectively rather than automatically.

## Final Position

This project should be presented as an interpretable multimodal analytics study of **normal pre-COVID local community attention dynamics**, not as a claim that the most complex model always wins.

The strongest academic story is:

- **Normal-period count patterns are learnable.** The all-modality HGB regressor wins the 2019 full-cohort test split, showing that richer features can help under stable conditions.
- **Temporal baselines remain important.** Rolling and last-month baselines are still strong references, especially for highly active businesses.
- **Attention pulses are a better framing for community attention.** They focus on unusual short-term shifts relative to each business's own recent baseline.
- **Sentiment is descriptively meaningful.** Pulse rows have higher VADER positivity and compound sentiment, so sentiment helps explain community enthusiasm even when it does not create large standalone model lift.
- **SNA and NLP add interpretation more than guaranteed lift.** Social exposure, recent language, VADER sentiment, and TF-IDF topic indicators help explain possible precursors, while selected models remain easier to defend than indiscriminately using every feature.
- **Case studies make the model auditable.** Correct predictions, missed pulses, and false alarms show where digital traces anticipate attention and where Yelp data remains incomplete.

Key limitations:

- Yelp friendship links are static.
- SNA features measure exposure, not causal influence.
- NLP features are lightweight VADER/lexicon/text-length/TF-IDF signals.
- This evaluation excludes the 2020-2021 disruption, so it should not be read as shock-robust forecasting.
- Review activity is a proxy for Yelp attention, not revenue or true customer volume.
- Static business metadata may include end-of-dataset information.
- Calibration is diagnostic, not a guarantee that probabilities transfer outside this dataset.